<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environment & Cost Hygiene

**Goal:** Set up API keys via Colab secrets, add spend guards, and make one successful model call.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


In [ ]:
%pip install -q anthropic

## API keys via Colab Secrets

Never paste an API key into a notebook cell. Notebooks get shared, committed, and screenshotted — a key in a cell is a key you'll be rotating next week.

Colab has a proper place for secrets:

1. Click the **key icon** in the left sidebar ("Secrets").
2. Click **Add new secret**. Name it exactly `ANTHROPIC_API_KEY`, paste your key as the value.
3. Flip the **Notebook access** toggle on for this notebook. Colab asks per notebook — that's the feature, not a bug. A shared copy of your notebook can't read your secret.

The cell below reads that secret in Colab, or falls back to the `ANTHROPIC_API_KEY` environment variable if you're running locally. Every notebook in this repo starts with the same cell, so once the secret exists you never think about it again.


In [ ]:
import os

# In Colab, read the key from Secrets (key icon in the left sidebar).
# Locally, set the ANTHROPIC_API_KEY env var instead.
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except ImportError:
    assert os.environ.get('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY'

import anthropic
client = anthropic.Anthropic()
MODEL = 'claude-sonnet-5'  # good default: capable and cheap enough to iterate on


## First call

One request, one response. Note the shape: `response.content` is a *list* of blocks (text, tool use, thinking, ...), not a bare string — check `block.type` before reading `block.text`. And `response.usage` is the billing record for this call; we'll build on it in a moment.


In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[{'role': 'user', 'content': 'In two sentences: what does a forward-deployed engineer do?'}],
)

for block in response.content:
    if block.type == 'text':
        print(block.text)

print()
print('stop_reason:', response.stop_reason)
print('usage:', response.usage)


## Reading `response.usage`

Every response carries the exact token accounting — no client-side estimation needed:

- `input_tokens` — prompt tokens billed at the full input rate
- `output_tokens` — everything the model generated (including any internal thinking)
- `cache_creation_input_tokens` / `cache_read_input_tokens` — prompt-cache activity; zero until you opt in (notebook 04 covers this)

The true prompt size is the sum of `input_tokens` + both cache fields. For cost, multiply each bucket by its rate. That's all a spend guard needs.


## A `spend_guard` for the session

During development you'll re-run cells, loop over datasets, and occasionally fat-finger a loop that makes 500 calls instead of 5. A small wrapper that accumulates estimated cost and refuses to go past a session budget turns that mistake from a bill into an exception.

Prices below are the public per-million-token rates as of mid-2026 — they drift, so treat the table as something you update, not a constant.


In [ ]:
PRICES = {
    # USD per million tokens: (input, output). Check the current pricing page — these drift.
    'claude-sonnet-5':  (3.00, 15.00),
    'claude-haiku-4-5': (1.00, 5.00),
    'claude-opus-4-8':  (5.00, 25.00),
}


class BudgetExceeded(Exception):
    pass


class SpendGuard:
    """Wraps client.messages.create, accumulates estimated cost, raises past a budget."""

    def __init__(self, budget_usd=0.50):
        self.budget = budget_usd
        self.spent = 0.0
        self.calls = 0

    def _cost(self, model, usage):
        inp_rate, out_rate = PRICES[model]
        # Ignoring cache pricing for now (exercise 3 fixes that).
        return (usage.input_tokens * inp_rate + usage.output_tokens * out_rate) / 1_000_000

    def create(self, **kwargs):
        if self.spent >= self.budget:
            raise BudgetExceeded(
                f'Session spend ${self.spent:.4f} >= budget ${self.budget:.2f} — refusing to call.'
            )
        response = client.messages.create(**kwargs)
        cost = self._cost(kwargs['model'], response.usage)
        self.spent += cost
        self.calls += 1
        print(
            f'[spend_guard] call {self.calls}: '
            f'{response.usage.input_tokens} in / {response.usage.output_tokens} out '
            f'~= ${cost:.5f}  (session: ${self.spent:.5f} / ${self.budget:.2f})'
        )
        return response


guard = SpendGuard(budget_usd=0.25)


In [ ]:
# Normal use: same signature as client.messages.create, plus a running tally.
resp = guard.create(
    model=MODEL,
    max_tokens=100,
    messages=[{'role': 'user', 'content': 'One sentence: why do teams add spend guards around LLM calls?'}],
)
print(resp.content[0].text)


In [ ]:
# And the failure path: a deliberately tiny budget trips after the first call.
tiny = SpendGuard(budget_usd=0.0001)
try:
    tiny.create(model=MODEL, max_tokens=50,
                messages=[{'role': 'user', 'content': 'Say hi.'}])  # this one succeeds...
    tiny.create(model=MODEL, max_tokens=50,
                messages=[{'role': 'user', 'content': 'Say hi again.'}])  # ...this one raises
except BudgetExceeded as e:
    print('Guard tripped as expected:', e)


Run the two cells above and note that the guard checks *before* calling — the second call never leaves your machine. In a real service you'd hang this off a per-tenant or per-day counter in Redis rather than an in-memory object, but the shape is identical: meter from `response.usage`, gate before the request.


## Picking a model (as of 2026)

The names will change; the principle won't. Three tiers:

- **Sonnet-class** (`claude-sonnet-5`) — the iterate-on default. Strong enough for almost everything, cheap enough that a day of experimenting costs pocket change.
- **Haiku-class** (`claude-haiku-4-5`) — bulk and cheap operations: classification, extraction over thousands of rows, judge calls in eval loops, anything where you multiply by N.
- **Opus/Fable-class** (`claude-opus-4-8`, `claude-fable-5`) — when reasoning depth actually matters: hard debugging, long-horizon agent runs, final-quality outputs. Noticeably more expensive per token; also note Fable-class pricing sits above Opus.

The working rule: **develop on cheap, eval on target.** Build the pipeline against Sonnet or Haiku, then run your evaluation set against the model you'll actually ship before you commit. Prompt behavior shifts between tiers — a prompt tuned on Haiku may over-trigger on Opus — so the eval-on-target step is not optional.


In [ ]:
# Same prompt on two tiers — compare cost and feel. (2 API calls.)
prompt = 'Classify the sentiment of this review as positive, negative, or mixed: '\
         '"The battery life is great but the screen scratched in a week." Reply with one word.'

for model in ['claude-haiku-4-5', 'claude-sonnet-5']:
    r = guard.create(model=model, max_tokens=10,
                     messages=[{'role': 'user', 'content': prompt}])
    print(f'{model}: {r.content[0].text.strip()}')


Run it and note that both tiers nail a task this simple — which is exactly the point. If Haiku passes your eval for a task, routing it to Opus is just burning margin.


## Exercises

1. Extend `SpendGuard.create` to record per-call wall-clock latency (`time.monotonic()` around the request) and add a `.report()` method that prints total cost, total calls, and p50/max latency.
2. Take a prompt you care about and phrase it two ways — terse bullet-point style vs. full prose. Compare `input_tokens` for each (send both with `max_tokens=1` to keep it cheap) and compute the cost difference at 100k calls/month.
3. Fix the guard's blind spot: include `cache_creation_input_tokens` (bill at 1.25x the input rate) and `cache_read_input_tokens` (0.1x) in `_cost`.
4. Add a `hard_max_tokens` option to `SpendGuard` that rejects any call requesting more than N output tokens, so a typo like `max_tokens=100000` can't slip through.
